# Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import joblib

from ml_fmda.moisture_rnn import OperationalRNNPredictor
from ml_fmda.utils import read_yml, Dict

In [ ]:
params = Dict(read_yml("tests/params_test.yaml"))

## Build a prediction model from params specification

The model class `OperationalRNNPredictor` supports cyclical prediction: the recurrent state can be saved after a call to `predict_cycle()` and used to initialize the state for the next call. This allows for real-time prediction as potentially large data comes in cycles (e.g. `wrfxpy`). 

The model class builds the model architecture from a structured dictionary that specifies features, hidden layers, units per layer, activation functions, etc. NOTE: as of Sept 2026, only support for 1 output dimension (the FMC), and time-warping only supports 1 LSTM layer (multi-layer has not been extensively validated).

We demonstrate functionality with random model weights. See `tutorial_operational.ipynb` for a demonstration with real data

In [ ]:
params

In [ ]:
rnn = OperationalRNNPredictor(params=params)
rnn.summary()

## Basic Predict

The standard `predict()` function behaves as a normal tensorflow model object. Input structure is `(nbatch, ntimes, nfeatures)`, and output is `(nbatch, ntimes, 1)`.

We demonstrate functionality with random data

In [ ]:
nbatch = 7
ntime = 24
nfeatures = 3

X = np.random.rand(nbatch, ntime, nfeatures)
print(f"{X.shape=}")

In [ ]:
preds = rnn.predict(X)
print(f"{preds.shape=}")

## Cyclical Predict

The `predict_cycle()` function has the `reset_state` argument. Sets the initial recurrent state to zeros if true. If false, it uses the internally stored recurrent state

In [ ]:
# Whole period prediction
preds_full = rnn.predict_cycle(X, reset_state=True)

# Cyclical prediction broken into 2
split = ntime // 2 # half the sequence times
preds1 = rnn.predict_cycle(X[:, :split, :], reset_state=True)
preds2 = rnn.predict_cycle(X[:, split:, :], reset_state=False)

preds_cycle = np.concatenate([preds1, preds2], axis=1)

print("Full prediction shape:", preds_full.shape)
print("Cycled prediction shape:", preds_cycle.shape)
print("Predictions match:", np.allclose(preds_cycle, preds_full))

## Saving Recurrent state

The `predict_cycle()` has argument `return_states`. When true, the model returns predictions and matrices of recurrent state variables. For one lstm layer, you expect matrices for the hidden state and cell state, both of shape `(nbatch, ntime)`. These can be passed to `predict_cycle()` with the `intial_state` argument.

In [ ]:
split = ntime // 2

preds_full = rnn.predict_cycle(X, reset_state=True)

preds1, states = rnn.predict_cycle(
    X[:, :split, :],
    reset_state=True,
    return_states=True,
)

preds2 = rnn.predict_cycle(
    X[:, split:, :],
    initial_states=states,
)

preds_cycle = np.concatenate([preds1, preds2], axis=1)

print("Number of recurrent states:", len(states))
print("State shapes [hidden, cell]:", [state.shape for state in states])
print("Predictions match:", np.allclose(preds_cycle, preds_full))